# Reviewer Response Analysis — Follow-up, Recurrence Timing & Verification of Reported Statistics

**Purpose.** This notebook supports the revision of *Craniopharyngioma.AI* in response to JNLS-B Skull Base reviewer comments (June 2026). It:

1. Documents **which dataset** the published analysis uses.
2. **Verifies** the descriptive and outcome statistics reported in the manuscript.
3. Computes the **follow-up duration, recurrence-free interval, and recurrence characterization** requested by Reviewer #1 (comment 4).

It does **not** retrain the models or modify any existing file. It only reads `data.csv` and prints/derives summary statistics.

---

### Which dataset is used?

The modeling and EDA notebooks (`Explainable.ipynb`, `EDA.ipynb`, `second.ipynb`, `third.ipynb`, `test.ipynb`) all load **`data.csv`** as the raw source. `data_clean.csv` and `data_imputed.csv` are *derived* products generated from `data.csv` (e.g., `start.ipynb` writes `data_clean.csv` via `df_clean.to_csv("data_clean.csv")`). Therefore `data.csv` is the canonical raw dataset and is used here.

In [1]:
import pandas as pd, numpy as np
df = pd.read_csv("data.csv")
print("Canonical dataset: data.csv")
print("Shape:", df.shape)
C = list(df.columns)
def col(name_substr):
    hits=[c for c in C if name_substr.lower() in c.lower()]
    return df[hits[0]] if hits else None

Canonical dataset: data.csv
Shape: (59, 59)


## 1. Cohort size and primary outcomes (verify manuscript)

In [2]:
n = len(df)
di  = pd.to_numeric(df["Permanent DI"], errors="coerce")
rec = pd.to_numeric(df["Recurrence after this operation"], errors="coerce")

di_n   = int((di==1).sum()); di_den = int(di.notna().sum())
rec_n  = int((rec==1).sum())
print(f"N patients                 : {n}            (paper: 59)")
print(f"Permanent DI               : {di_n}/{di_den} = {100*di_n/di_den:.1f}%   (paper: 38, 65.5%)")
print(f"Tumor recurrence           : {rec_n}/{n} = {100*rec_n/n:.1f}%   (paper: 17, 28.8%)")

N patients                 : 59            (paper: 59)
Permanent DI               : 38/58 = 65.5%   (paper: 38, 65.5%)
Tumor recurrence           : 17/59 = 28.8%   (paper: 17, 28.8%)


## 2. Demographics (verify manuscript)

In [3]:
age = pd.to_numeric(df["Age "], errors="coerce")
bmi = pd.to_numeric(df["BMI"], errors="coerce")
print(f"Age  : mean {age.mean():.1f} +/- {age.std():.1f}, median {age.median():.0f}, "
      f"IQR {age.quantile(.75)-age.quantile(.25):.0f}, range {age.min():.0f}-{age.max():.0f}")
print("        (paper: 45.4 +/- 17.9, median 48, IQR 27)")
print(f"BMI  : mean {bmi.mean():.1f} +/- {bmi.std():.1f} kg/m^2 (n={bmi.notna().sum()})   (paper: 28.0 +/- 6.7)")

Age  : mean 45.4 +/- 17.9, median 48, IQR 27, range 9-76
        (paper: 45.4 +/- 17.9, median 48, IQR 27)
BMI  : mean 28.0 +/- 6.7 kg/m^2 (n=55)   (paper: 28.0 +/- 6.7)


## 3. Surgical, resection, histology and prior-operation breakdown (verify manuscript)

In [4]:
print("Surgical approach :", df["Surgical Approach"].value_counts(dropna=False).to_dict(),
      "  (paper: endoscopic 30, craniotomy 29)")
print("Resection outcome :", df["Resection outcome (total, subtotal, partial)"].value_counts(dropna=False).to_dict(),
      "  (paper: total 18, subtotal 30, near total 10)")
print("Recurring (prior) :", int((pd.to_numeric(df["recurring tumor?"],errors='coerce')==1).sum()),
      "  (paper: 15, 25.4%)")
print("Histology         :", df["Tumor (Adamantinomatous, papillary)"].value_counts(dropna=False).to_dict())

Surgical approach : {'endoscopic': 30, 'craniotomy': 29}   (paper: endoscopic 30, craniotomy 29)
Resection outcome : {'subtotal': 30, 'total': 18, 'near total': 10, nan: 1}   (paper: total 18, subtotal 30, near total 10)
Recurring (prior) : 15   (paper: 15, 25.4%)
Histology         : {'Adamantinomatous ': 28, nan: 20, 'Papillary': 11}


## 4. Follow-up duration  *(Reviewer #1, comment 4)*

The stored `Length of followup` column contains data-entry errors (negative values and a `9/25/1902` placeholder date in two rows). Follow-up is therefore recomputed directly from the surgery `Date` and `Most recent follow up` columns, excluding the invalid placeholder rows.

In [5]:
surg = pd.to_datetime(df["Date"], errors="coerce")
last = pd.to_datetime(df["Most recent follow up"], errors="coerce")
fu_days = (last - surg).dt.days
invalid = fu_days[fu_days < 0]
print("Rows with invalid (negative) follow-up excluded:", invalid.index.tolist(),
      "-> last-FU date =", df.loc[invalid.index, "Most recent follow up"].tolist())
fu_mo = (fu_days[fu_days >= 0]) / 30.44
print(f"\nValid follow-up: n = {fu_mo.notna().sum()}")
print(f"Median follow-up = {fu_mo.median():.1f} months  ({fu_mo.median()/12:.1f} years)")
print(f"IQR              = {fu_mo.quantile(.25):.1f} - {fu_mo.quantile(.75):.1f} months")
print(f"Range            = {fu_mo.min():.1f} - {fu_mo.max():.1f} months")

Rows with invalid (negative) follow-up excluded: [49, 55] -> last-FU date = ['9/25/1902', '9/25/1902']

Valid follow-up: n = 54
Median follow-up = 28.1 months  (2.3 years)
IQR              = 7.9 - 60.3 months
Range            = 0.6 - 173.0 months


## 5. Recurrence characterization & time-to-recurrence  *(Reviewer #1, comment 4)*

Recurrence is recorded radiographically (date taken from surveillance imaging in `Recurrence date` / `Days to recurrence`). We report the time from index surgery to recurrence and how many recurrences were treated (salvage re-operation and/or radiation).

In [6]:
d2r_mo = pd.to_numeric(df["Days to recurrence"], errors="coerce") / 30.44
r = d2r_mo[rec==1]
print(f"Recurrences                              : {int((rec==1).sum())}")
print(f"Recurrences with documented date         : {int(r.notna().sum())} / {int((rec==1).sum())} (radiographically detected)")
print(f"Median time-to-recurrence                : {r.median():.1f} months")
print(f"Range time-to-recurrence                 : {r.min():.1f} - {r.max():.1f} months")

reop = df.loc[rec==1, "Re-operation after this operation (add date)"].notna().sum()
radp = (pd.to_numeric(df.loc[rec==1, "Radiation post (y/n)"], errors="coerce")==1).sum()
print(f"Recurrences treated with salvage re-op   : {int(reop)}")
print(f"Recurrences treated with radiation       : {int(radp)}")

fu_nonrec = ((last - surg).dt.days[(rec==0)] / 30.44)
fu_nonrec = fu_nonrec[fu_nonrec>=0]
print(f"\nMedian follow-up among non-recurrers (recurrence-free): {fu_nonrec.median():.1f} months")

Recurrences                              : 17
Recurrences with documented date         : 17 / 17 (radiographically detected)
Median time-to-recurrence                : 10.2 months
Range time-to-recurrence                 : 1.1 - 113.1 months
Recurrences treated with salvage re-op   : 8
Recurrences treated with radiation       : 10

Median follow-up among non-recurrers (recurrence-free): 28.3 months


## 6. Summary for the manuscript / response letter

| Statistic | Value |
|---|---|
| Patients | 59 |
| Permanent DI | 38/58 (65.5%) |
| Recurrence | 17/59 (28.8%) |
| Median follow-up | 28.1 months (IQR 7.9-60.3; range 0.6-173.0) |
| Recurrence detection | radiographic (surveillance MRI), all 17 dated |
| Median time-to-recurrence | 10.2 months (range 1.1-113.1) |
| Recurrences re-operated | 8 |
| Recurrences irradiated | 10 |

**Caveat:** recurrence is an inherently time-to-event outcome; here it is modeled as a binary endpoint with variable/incomplete follow-up. These descriptive figures are reported in the revised Methods/Results, and formal survival modeling in a larger cohort is noted as a needed next step.